**生成 dws_season_rating_comparison表**

In [ ]:
#安装特定库
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.2 MB/s eta 0:00:00


In [ ]:
#加载特定库
import duckdb
import pandas as pd
from pymongo import MongoClient

#连接MongoDB
try:
  client=MongoClient("mongodb+srv://@cluster0.mqjqhzt.mongodb.net/?appName=Cluster0")
  db_ods=client["ods_data"]

  print("连接成功")
except BaseException:
  print("连接失败")

#读取ods层数据表
collection_ods_comment=db_ods["ods_douban_comment"]
collection_ods_douban_rating=db_ods["ods_douban_season_rating"]
collection_ods_imdb_rating=db_ods["ods_friends_episodes_v3"]

#转换表格形式
df_ods_comment=pd.DataFrame(list(collection_ods_comment.find({},{"_id":0})))
df_ods_douban_rating=pd.DataFrame(list(collection_ods_douban_rating.find({},{"_id":0})))
df_ods_imdb_rating=pd.DataFrame(list(collection_ods_imdb_rating.find({},{"_id":0})))

#生成 dws_season_rating_comparison表
df_dws_rating=duckdb.query("""
  SELECT d.season,
      d.rating AS df_douban_rating,
      i.imdb_rating,
      ROUND((d.rating-i.imdb_rating),1) AS rating_difference
  FROM df_ods_douban_rating d
  LEFT JOIN (SELECT ROUND(AVG(Stars),1) AS imdb_rating,
             Season
         FROM df_ods_imdb_rating
         GROUP BY Season) i
  ON d.season=i.Season
  ORDER BY d.season
""").df()

#生成 dws_douban_comment
df_dws_comment=duckdb.query("""
  SELECT id AS user_id,
      season,
      "name" AS user_name,
      comment
  FROM df_ods_comment
  ORDER BY season
""").df()

#测试查验
print(df_dws_rating.head())
print(df_dws_comment.head())

#导入MongoDB
#1.连接数据库及数据表
try:
  db_dws=client["dws_data"]
except BaseException:
  print("连接失败")

try:
  db_dws["dws_season_rating_comparison"].insert_many(df_dws_rating.to_dict("records"))
  db_dws["dws_douban_comment"].insert_many(df_dws_comment.to_dict("records"))
except BaseException:
  print("导入失败")






连接成功
   season  df_douban_rating  imdb_rating  rating_difference
0       1               9.7          8.3                1.4
1       2               9.8          8.5                1.3
2       3               9.7          8.4                1.3
3       4               9.7          8.5                1.2
4       5               9.8          8.6                1.2
   user_id  season user_name                  comment
0      199       1     猫猫eko   这个就不用我多说了吧。。。不笑除非你没长嘴。
1      350       1      日光瀑布                   我的大学时光
2       34       1     赫舍里清如      三刷，全程都在笑，只是第三遍有些带泪了
3      209       1        戈戈  第八季后有点走下坡了。朋友朋友朋友朋友，呵呵。
4       94       1      松鼠排队        真老，真差，真低IQ，真妄得虚名！
